In [1]:
# Preparation
import pandas as pd, numpy as np, sqlalchemy as sqla, shutil, decouple
from sqlalchemy import String
from IPython.display import display

engine = sqla.create_engine(decouple.config("MIS_DB"))
mis_file_path = decouple.config("MIS_FILE_PATH")

pd.set_option("display.max_rows", None)
pd.set_option("display.float_format", "{:.2f}".format)

# Sales per Product data preparation
sku_df = pd.read_excel(f"{mis_file_path}MIS Data\\Sell-In\\Raw Data\\Consignment\\Product Sales (CN).xlsx", sheet_name="Sheet1", parse_dates=["Date"], dtype={
    "Unnamed:0": "str",
    "Unnamed:1": "str",
    "Unnamed:2": "str",
    "Type": "str",
    "Unnamed:4": "str",
    "Unnamed:6": "str",
    "P.O.#": "str",
    "Unnamed:8": "str",
    "Name": "str",
    "Unnamed:10": "str",
    "Num": "str",
    "Unnamed:12": "str",
    "Memo": "str",
    "Unnamed:14": "str",
    "Item": "str",
    "Unnamed:16": "str",
    "ItemDescription": "str",
    "Unnamed:18": "str",
    "Account": "str",
    "Unnamed:20": "str",
    "Class": "str",
    "Unnamed:22": "str",
    "Rep": "str",
    "Unnamed:24": "str",
    "Qty": "float64",
    "Unnamed:26": "str",
    "U/M": "str",
    "Unnamed:28": "str",
    "Amount": "float64",
    "Unnamed:30": "str",
    "SalesTaxCode": "str",
    "Unnamed:32": "str",
    "Inventory Site": "str",
    "Unnamed:34": "str",
    "ShipToAddress1": "str",
    "Unnamed:36": "str",
    "ShipToAddress2": "str"
})

for col in sku_df.columns:
    if "Unnamed: " in col:
        sku_df = sku_df.drop(col, axis=1, inplace=False)

sku_df = sku_df.drop(sku_df.index[0], inplace=False)
sku_df = sku_df.drop(sku_df.index[-1], inplace=False)

# Monthly Sales data preparation
mnth_df = pd.read_excel(f"{mis_file_path}MIS Data\\Sell-In\\Raw Data\\Consignment\\Monthly Sales (CN).xlsx", sheet_name="Sheet1", parse_dates=["Date"], dtype={
    "Unnamed:0": "str",
    "Unnamed:1": "str",
    "Unnamed:2": "str",
    "Type": "str",
    "Unnamed:4": "str",
    "Unnamed:6": "str",
    "Num": "str",
    "Unnamed:8": "str",
    "Name": "str",
    "Unnamed:10": "str",
    "Class": "str",
    "Unnamed:12": "str",
    "Amount": "float64",
    "Unnamed:14": "str",
    "Memo": "str",
    "Unnamed:16": "str",
    "ShipToAddress1": "str",
    "Unnamed:18": "str",
    "ShipToAddress2": "str"
})

for col in mnth_df.columns:
    if "Unnamed: " in col:
        mnth_df = mnth_df.drop(col, axis=1, inplace=False)

mnth_df = mnth_df.drop(mnth_df.index[0], inplace=False)
mnth_df = mnth_df.drop(mnth_df.index[-1], inplace=False)
# mnth_df

In [ ]:
# Filter included invoices in monthly sales
consignment_accs = (
    "Cash & Carry Department Store", 
    "Ever Commonwealth Center Inc. - DS", 
    # "Ever Supermarket",
    "Gaisano Capital Super Market",
    "Gaisano Grand",
    # "LCC Department Store",
    # "LCC Supermarket",
    # "Metro Gaisano",
    # "Metro Retail Stores Group, Inc.",
    "Sta. Lucia East Department Store, Inc.",
    "Sta. Lucia East Supermarket",
    "Tropical Hut Super Market",
    "UM Superfoods Corp.",
    "Unimart, Incorporated",
    # "Waltermart Supermarket, Inc."
)

month_names = (
    "January", "February", "March", "April", "May", "June",
    "July", "August", "September", "October", "November", "December"
)

mnth_df["is_included"] = mnth_df["Name"].str.contains("|".join(consignment_accs), na=False, case=False)
mnth_df["month"] = mnth_df["Memo"].str.extract("(" + "|".join(month_names) + ")")

#Save monthly sales with is_included
mnthly_data = mnth_df

mnth_net = mnth_df[mnth_df["is_included"]]
mnth_net = mnth_net.groupby("Num")["Amount"].sum()
mnth_net = mnth_net.to_frame().reset_index()
mnth_net = mnth_net.rename(columns={"Amount":"month_net"})

mnth_net
# mnth_df.head()

,Num,month_net
0,39535,5880.00
1,39536,1045.50
2,93813,5661.75
3,93819,3651.00
4,94272,1156.01
5,94301,694.40
6,94302,12303.58
7,94304,3486.69


In [3]:
# Filter included invoices in sales per product based on monthy sales
display(sku_df.head())

sku_df = pd.merge(sku_df, mnth_df[["Num", "Name", "month", "is_included"]], on=["Num", "Name"], how="left")

# display(sku_df.head())
# sku_df = sku_df[sku_df["is_included"]]
sku_df = sku_df[sku_df["is_included"].fillna(False)]

# Add column for discount rows
sku_df["is_disc"] = sku_df["Item"].str.contains("Discount")
# sku_df

# Group by discount
group = 0
groups = []
for i in range(len(sku_df)):
    current = sku_df.iloc[i]["is_disc"]
    if i == 0:
        groups.append(group)
    else:
        previous = sku_df.iloc[i - 1]["is_disc"]
        if previous and not current:
            group = group + 1
        groups.append(group)
sku_df["group_num"] = groups

# Add group_name column (needed for groupby)
sku_df["group_name"] = sku_df["group_num"].astype(str) + "-" + sku_df["Name"] + "-" + sku_df["Num"]

,Type,Date,P. O. #,Name,Num,Memo,Item,Item Description,Account,Class,Rep,Qty,U/M,Amount,Sales Tax Code,Inventory Site,Ship To Address 1,Ship To Address 2
1,Invoice,2026-05-30,0065813432-00,Puregold Price Club Inc.,94633,iWhite Korea Grand Korean Sale 2026 B2T2 - Fac...,Assembled Items:GS 2026 - B2G2 FCS,iWhite Korea Grand Korean Sale 2026 B2T2 - Fac...,Skin Care,Skin Care:NCR,JA,1.00,Pack of 6,267.86,Taxable Sales,FG Warehouse,PPCI - BACOLOD (OLSI) (01041),Manila
2,Invoice,2026-05-30,0065813432-00,Puregold Price Club Inc.,94633,iWhite Korea Grand Korean Sale 2026 B2T2 - Aqu...,Assembled Items:GS 2026 - B2G2 PACS,iWhite Korea Grand Korean Sale 2026 B2T2 - Aqu...,Skin Care,Skin Care:NCR,JA,1.00,Pack of 6,289.29,Taxable Sales,FG Warehouse,PPCI - BACOLOD (OLSI) (01041),Manila
3,Invoice,2026-05-30,0065813432-00,Puregold Price Club Inc.,94633,Sales Discount,Sales Discount:30%,Sales Discount,Sales Discount,Skin Care:NCR,JA,NaN,NaN,-167.15,Taxable Sales,NaN,PPCI - BACOLOD (OLSI) (01041),Manila
4,Invoice,2026-05-30,0065813429-00,Puregold Price Club Inc.,94634,iWhite Korea Grand Korean Sale 2026 B2T2 - Fac...,Assembled Items:GS 2026 - B2G2 FCS,iWhite Korea Grand Korean Sale 2026 B2T2 - Fac...,Skin Care,Skin Care:NCR,JA,1.00,Pack of 6,267.86,Taxable Sales,FG Warehouse,PPCI Long Haul (North) (01026),Manila
5,Invoice,2026-05-30,0065813429-00,Puregold Price Club Inc.,94634,Sales Discount,Sales Discount:30%,Sales Discount,Sales Discount,Skin Care:NCR,JA,NaN,NaN,-80.36,Taxable Sales,NaN,PPCI Long Haul (North) (01026),Manila


In [4]:
# Multiplied Discount by group
# Strip discount amount from string and convert to decimal
sku_df["mltpld_disc"] = np.where(
    sku_df["is_disc"],
    sku_df["Item"].str[15:20].str.strip("%"),
    0
)

sku_df["mltpld_disc"] = np.where(
    sku_df["is_disc"],
    1 - (sku_df["mltpld_disc"].astype(float) / 100),
    0
)

# Filter discount rows, groupby multiplication, convert to df and reset index
mltpld_disc_srs = sku_df[sku_df["is_disc"]]
mltpld_disc_srs = mltpld_disc_srs.groupby("group_name")["mltpld_disc"].prod()
mltpld_disc_df = mltpld_disc_srs.to_frame().reset_index()

In [5]:
# Total Discount per group
# Clean start
sku_df = sku_df.drop(columns=["mltpld_disc"])

# Strip discount amount from string and convert to decimal
sku_df["total_disc"] = np.where(
    sku_df["is_disc"],
    sku_df["Item"].str[15:20].str.strip("%"),
    0
)
sku_df["total_disc"] = sku_df["total_disc"].astype(float) / 100

# Create grouped discount series, convert to df and reset index
grp_disc_srs = 1 - sku_df.groupby("group_name")["total_disc"].sum()
grp_disc_df = grp_disc_srs.to_frame().reset_index()

In [6]:
# Initial matching with Monthly Sales using grouped discount
# Remove discount rows
sku_df = sku_df[sku_df["is_disc"] == False]

# Clean start for matching
rmv_cols =["is_included", "is_disc", "group_num", "total_disc"]
sku_df = sku_df.drop(columns=rmv_cols)

# Add net_1 column
sku_df = sku_df.merge(grp_disc_df, on="group_name", how="left")
sku_df["Net Amount"] = np.where(
    sku_df["Sales Tax Code"] == "Taxable Sales",
    sku_df["Amount"] * sku_df["total_disc"] * 1.12,
    sku_df["Amount"] * sku_df["total_disc"]
)

# Sum of net_1 per invoice
chck_net_1 = sku_df.groupby("Num")["Net Amount"].sum()
chck_net_1 = chck_net_1.to_frame().reset_index()

# Compare Sales per Product and Monthly sales per invoice
chck_net_1 = chck_net_1.merge(mnth_net, on="Num", how="left")

# Filter matched invoices
chck_net_1["is_same"] = np.where((chck_net_1["Net Amount"] - chck_net_1["month_net"]).abs() < 1, True, False)
sku_df = pd.merge(sku_df, chck_net_1[["Num", "is_same"]], on="Num", how="left")

# Save matched results
match_net_1 = sku_df[sku_df["is_same"]]
match_net_1 = match_net_1.drop(columns="total_disc")

In [7]:
# Second matching with Monthly Sales using invoice discount and mismatched_data from initial matching
# Use mismatched_data data from previous step
sku_df = sku_df[~sku_df["is_same"]]

# Clean start for matching
rmv_cols = ["total_disc", "Net Amount", "is_same"]
sku_df = sku_df.drop(columns=rmv_cols)

# Add net_2 column
sku_df = sku_df.merge(mltpld_disc_df, on="group_name", how="left")
sku_df["Net Amount"] = np.where(
    sku_df["Sales Tax Code"] == "Taxable Sales",
    sku_df["Amount"] * sku_df["mltpld_disc"] * 1.12,
    sku_df["Amount"] * sku_df["mltpld_disc"]
)

# Sum of net_2 per invoice
chck_net_2 = sku_df.groupby("Num")["Net Amount"].sum()
chck_net_2 = chck_net_2.to_frame().reset_index()

# Compare Sales per Product and Monthly sales per invoice
chck_net_2 = chck_net_2.merge(mnth_net, on="Num", how="left")

# Filter matched invoices
chck_net_2["is_same"] = np.where(
    (chck_net_2["Net Amount"] - chck_net_2["month_net"]).abs() < 1,
    True,
    False
)

sku_df = pd.merge(sku_df, chck_net_2[["Num", "is_same"]], on="Num", how="left")

match_net_2 = sku_df[sku_df["is_same"]]
match_net_2 = match_net_2.drop(columns="mltpld_disc")

mismatched_data = sku_df[~sku_df["is_same"]]

matched_data = pd.concat([match_net_1, match_net_2], ignore_index=True)

rmv_cols = ["group_name", "is_same"]
matched_data = matched_data.drop(columns=rmv_cols)

# Save checker file as xlsx
xl_writer = pd.ExcelWriter("C:\\MIS Outputs\\Checker.xlsx", engine="xlsxwriter")
mismatched_data.to_excel(xl_writer, sheet_name="Mismatch", float_format="%.2f", index=False)
matched_data.to_excel(xl_writer, sheet_name="Sales per Product", float_format="%.2f", index=False)
mnthly_data.to_excel(xl_writer, sheet_name="Monthly Sales", float_format="%.2f", index=False)
xl_writer.close()

matched_data = matched_data.rename(columns={
    "Type": "type",
    "Date": "date",
    "P. O. #": "po_num",
    "Name": "name",
    "Num": "num",
    "Memo": "memo",
    "Item": "item",
    "Item Description": "item_desc",
    "Account": "account",
    "Class": "class",
    "Rep": "rep",
    "Qty": "qty",
    "U/M": "um",
    "Amount": "amount",
    "Sales Tax Code": "sales_tax_code",
    "Inventory Site": "inventory_site",
    "Ship To Address 1": "ship_to_address_1",
    "Ship To Address 2": "ship_to_address_2",
    "Net Amount": "net_amount"
})

matched_data.to_sql(name="cn_qb_mtd", con=engine, schema="staging", if_exists="replace", index=False, 
    dtype={col: String(255) for col in matched_data.select_dtypes(include="object").columns}
)

month_num_arr = {
    "January": "01", "February": "02", "March": "03", "April": "04", "May": "05", "June": "06",
    "July": "07", "August": "08", "September": "09", "October": "10", "November": "11", "December": "12"
}

display(matched_data.head())
# matched_data.iloc[0, 17]

# year_month = matched_data["date"].max().strftime("%Y") + " " + month_num_arr.get(matched_data.iloc[0, 17]) + " " + matched_data.loc[0, 17]
# month_year = matched_data.iloc[0, 17] + " " + matched_data["date"].max().strftime("%Y")

# changed matched_data.iloc[0, 17] to matched_data.loc[0, "month"]
year_month = matched_data["date"].max().strftime("%Y") + " " + month_num_arr.get(matched_data.loc[0, "month"]) + " " + matched_data.loc[0, "month"]
month_year = matched_data.loc[0, "month"] + " " + matched_data["date"].max().strftime("%Y")

shutil.copy(f"{mis_file_path}MIS Data\\Sell-In\\Raw Data\\Consignment\\Product Sales (CN).xlsx", 
    f"{mis_file_path}MIS Data\\Sell-In\\Raw Data\\Consignment\\Compiled QB Extracted\\{year_month} Product Sales (CN).xlsx")

shutil.copy(f"{mis_file_path}MIS Data\\Sell-In\\Raw Data\\Consignment\\Monthly Sales (CN).xlsx",
    f"{mis_file_path}MIS Data\\Sell-In\\Raw Data\\Consignment\\Compiled QB Extracted\\{year_month} Monthly Sales (CN).xlsx")

matched_data.to_csv(f"{mis_file_path}MIS Data\\Sell-In\\Net\\Net Consignment as of {month_year}.csv", index=False)

,type,date,po_num,name,num,memo,item,item_desc,account,class,rep,qty,um,amount,sales_tax_code,inventory_site,ship_to_address_1,ship_to_address_2,month,net_amount
0,Invoice,2026-05-26,NaN,Gaisano Capital Super Market,93813,iWhite Korea Whitening Vita Aqua Moisturizer -...,Unpacked Products:Tube:BACT,iWhite Korea Whitening Vita Aqua Moisturizer -...,Skin Care,Skin Care:Visayas,NP,1.00,Tube,213.39,Taxable Sales,Taipan Dev. - Mactan,"Tai-pan Development, Inc. - Mactan Smkt","Pajo, Lapu-Lapu City",February,179.25
1,Invoice,2026-05-26,NaN,Gaisano Capital Super Market,93813,iWhite Korea Whitening Vita Facial Cream - 65m...,Unpacked Products:Tube:FCT,iWhite Korea Whitening Vita Facial Cream - 65m...,Skin Care,Skin Care:Visayas,NP,7.00,Tube,1493.75,Taxable Sales,Taipan Dev. - Mactan,"Tai-pan Development, Inc. - Mactan Smkt","Pajo, Lapu-Lapu City",February,1254.75
2,Invoice,2026-05-26,NaN,Gaisano Capital Super Market,93813,iWhite Korea Whitening Vita Facial Wash - 90ml...,Unpacked Products:Tube:BFWT,iWhite Korea Whitening Vita Facial Wash - 90ml...,Skin Care,Skin Care:Visayas,NP,5.00,Tube,977.68,Taxable Sales,Taipan Dev. - Mactan,"Tai-pan Development, Inc. - Mactan Smkt","Pajo, Lapu-Lapu City",February,821.25
3,Invoice,2026-05-26,NaN,Gaisano Capital Super Market,93813,iWhite Korea The Original Nose Pack - 3.5ml B...,Assembled Items:NPS Box 24's,iWhite Korea The Original Nose Pack - 3.5ml B...,Skin Care,Skin Care:Visayas,NP,16.00,PCS,285.71,Taxable Sales,Taipan Dev. - Mactan,"Tai-pan Development, Inc. - Mactan Smkt","Pajo, Lapu-Lapu City",February,240.00
4,Invoice,2026-05-26,NaN,Gaisano Capital Super Market,93813,iWhite Korea Whitening Pack - 8ml Box of 24's...,Assembled Items:WPS Box 24's,iWhite Korea Whitening Pack - 8ml Box of 24's...,Skin Care,Skin Care:Visayas,NP,5.00,PCS,111.61,Taxable Sales,Taipan Dev. - Mactan,"Tai-pan Development, Inc. - Mactan Smkt","Pajo, Lapu-Lapu City",February,93.75


In [8]:
# SQL Transformation

transform_query = """
    SELECT
        STR_TO_DATE(CONCAT(cn.`month`, "-1-",YEAR(cn.`date`)), "%M-%d-%Y") AS `date`, YEAR(cn.`date`) AS `year`, cn.`month`, "Sell-Out" AS `type`,
        cn.num, cn.po_num, cn.inventory_site, ref_an.account_name, ref_ad.account_chain, cn.ship_to_address_1, cn.ship_to_address_2, cn.rep, ref_ad.sales_group,
        ref_l.lead_name, ref_ad.bpc_region, ref_ad.account_channel, ref_ad.channel_class, ref_ad.business_unit, ref_ad.account_type, cn.item,
        ref_pd.product_name, ref_pc.product_code, ref_pd.brand, ref_pd.product_class, ref_pd.`usage` AS product_usage, ref_pd.product_type,
        ref_pd.product_category, cn.um, qty, cn.qty AS qty_pcs, cn.amount, cn.net_amount
    FROM (
        SELECT *, ROW_NUMBER() OVER() AS id FROM sales.cn_qb AS sub
    ) cn

    -- Add corrected name
    LEFT JOIN ref.account_names AS ref_an
        ON cn.`name` = ref_an.`name`

    -- Add account details
    LEFT JOIN ref.account_details AS ref_ad
        ON ref_an.account_name = ref_ad.account_name

    -- Add assigned team leader
    LEFT JOIN ref.lead_names AS ref_l
        ON ref_ad.lead_id = ref_l.lead_id
            AND YEAR(cn.`date`) = ref_l.`year`
            AND cn.`month` = ref_l.`month`

    -- Add product code
    LEFT JOIN ref.product_codes AS ref_pc
        ON cn.item = ref_pc.item

    -- Add product details
    LEFT JOIN ref.product_details AS ref_pd
        ON ref_pc.product_code = ref_pd.product_code1
"""

product_name_checker_query = """
    SELECT DISTINCT YEAR(s.`date`) AS `year`, s.`month`, s.item, ref_pc.product_code, ref_pd.product_name
    FROM staging.cn_qb_mtd AS s
    
    LEFT JOIN ref.product_codes AS ref_pc
        ON s.item = ref_pc.item
        
    LEFT JOIN ref.product_details AS ref_pd
        ON ref_pc.product_code = ref_pd.product_code1
        
    WHERE ref_pc.product_code IS NULL OR ref_pd.product_name IS NULL
"""

acc_name_checker_query = """
    SELECT DISTINCT YEAR(s.`date`) AS `year`, s.`month`, s.`name`, s.class, ref_an.`name`, ref_ad.account_chain
    FROM staging.cn_qb_mtd AS s
    
    LEFT JOIN ref.account_names AS ref_an
        ON s.`name` = ref_an.`name`
        
    LEFT JOIN ref.account_details AS ref_ad
        ON ref_an.account_name = ref_ad.account_name
        
    WHERE ref_an.`name` IS NULL OR ref_ad.account_chain IS NULL
"""

sell_in_df = pd.read_sql(sql=transform_query, con=engine)
product_name_df = pd.read_sql(sql=product_name_checker_query, con=engine)
account_name_df = pd.read_sql(sql=acc_name_checker_query, con=engine)

xl_writer = pd.ExcelWriter(f"C:\\MIS Outputs\\Consignment as of {month_year}.xlsx", engine="xlsxwriter")
sell_in_df.to_excel(xl_writer, sheet_name="Consignment", float_format="%.2f", index=False)
product_name_df.to_excel(xl_writer, sheet_name="Product Name", float_format="%.2f", index=False)
account_name_df.to_excel(xl_writer, sheet_name="Account Name", float_format="%.2f", index=False)
xl_writer.close()